# AWS Lake Formation — Fine-grained Access Control for Data Lakes

## Mental Model

Lake Formation is the governance control plane that sits on top of your S3-based data lake and AWS Glue Data Catalog.

For this notebook, keep the model simple:

- **S3** stores the files.
- **Glue Catalog** describes databases, tables, and columns.
- **Lake Formation** decides **who can see what** at the database, table, column, and row level.
- **IAM** still authenticates principals and allows them to call AWS APIs, but **Lake Formation** governs fine-grained access to registered lake resources.

### Business framing

The telemetry narrative for this notebook is a Citi-style observability estate:

- `endpoints`: 10,000 rows
- `metrics`: 500,000 rows
- `alerts`: 25,000 rows
- 6,000+ API endpoints monitored for latency, error rate, throughput
- alerting routed by severity tiers

In practice, those tables would be landed into S3 zones and surfaced in Glue/Lake Formation as governed lake tables.

This notebook focuses on:

1. Lake registration
2. Database/table permissions
3. Column-level security
4. Row-level filters
5. LF vs IAM decisioning

In [ ]:
# Standard library + AWS SDK only. Packages are assumed pre-installed per environment instructions.
import json
import time
import uuid
import traceback
from pprint import pprint

import boto3
from botocore.exceptions import ClientError, ProfileNotFound, BotoCoreError

In [ ]:
# ---- Runtime configuration ----
AWS_PROFILE = "study"
AWS_REGION = "us-east-1"
AWS_ACCOUNT_ID = "357811130281"

POSTGRES_CONTEXT = {
    "host": "localhost",
    "port": 5432,
    "database": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
    "tables": {
        "endpoints": {
            "rows": 10000,
            "columns": ["endpoint_id", "name", "region", "status", "category"],
        },
        "metrics": {
            "rows": 500000,
            "columns": ["endpoint_id", "metric_name", "value", "timestamp"],
        },
        "alerts": {
            "rows": 25000,
            "columns": ["alert_id", "endpoint_id", "severity", "message", "created_at"],
        },
    },
    "narrative": "6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers",
}

GLUE_DATABASE = "citi_deep_glue"
GLUE_TABLE = "alerts"

# Demo S3 path for Lake Formation registration. The code will try to create it if possible.
DEMO_BUCKET = f"citi-de-lf-demo-{AWS_ACCOUNT_ID}-{AWS_REGION}"
DEMO_PREFIX = "telemetry"
DEMO_S3_ARN = f"arn:aws:s3:::{DEMO_BUCKET}"
DEMO_S3_URI = f"s3://{DEMO_BUCKET}/{DEMO_PREFIX}/"

# Demo principals for grant examples.
SECURITY_ANALYST_ROLE_ARN = f"arn:aws:iam::{AWS_ACCOUNT_ID}:role/citi-security-analyst"
DATA_ENGINEER_ROLE_ARN = f"arn:aws:iam::{AWS_ACCOUNT_ID}:role/citi-data-engineer"

print("Configured account:", AWS_ACCOUNT_ID)
print("Configured region :", AWS_REGION)
print("Demo lake location:", DEMO_S3_URI)
print("Glue database/table:", f"{GLUE_DATABASE}.{GLUE_TABLE}")

In [ ]:
# ---- Session / client bootstrap ----
def safe_boto3_session(profile_name: str, region_name: str):
    """
    Build a boto3 session from the requested profile.
    Falls back to default credential chain if the named profile is unavailable.
    """
    try:
        return boto3.Session(profile_name=profile_name, region_name=region_name), {
            "profile_used": profile_name,
            "fallback": False,
        }
    except ProfileNotFound:
        return boto3.Session(region_name=region_name), {
            "profile_used": "default credential chain",
            "fallback": True,
        }

session, session_meta = safe_boto3_session(AWS_PROFILE, AWS_REGION)

lf = session.client("lakeformation", region_name=AWS_REGION)
glue = session.client("glue", region_name=AWS_REGION)
s3 = session.client("s3", region_name=AWS_REGION)
sts = session.client("sts", region_name=AWS_REGION)
iam = session.client("iam", region_name=AWS_REGION)

caller_identity = {}
try:
    caller_identity = sts.get_caller_identity()
except Exception as e:
    caller_identity = {"error": f"{type(e).__name__}: {e}"}

print("Session meta:")
pprint(session_meta)
print("\nCaller identity:")
pprint(caller_identity)

## Setup Notes

The notebook uses a `boto3` Lake Formation client built from the AWS profile `study` in `us-east-1`.

The code is written to be **operationally safe**:

- it uses real account/profile/region values from the environment spec,
- it wraps mutating AWS calls in error handling,
- and it records what succeeded, skipped, or failed without crashing the notebook.

That keeps the notebook runnable top-to-bottom even if a specific resource is missing or your principal lacks permission for one of the demo operations.

In [ ]:
# ---- Helper utilities ----
def call_aws(label, fn, *args, **kwargs):
    """
    Execute an AWS SDK call and return a structured result without raising uncaught exceptions.
    """
    try:
        response = fn(*args, **kwargs)
        return {
            "label": label,
            "ok": True,
            "response": response,
            "error": None,
        }
    except Exception as e:
        return {
            "label": label,
            "ok": False,
            "response": None,
            "error": f"{type(e).__name__}: {e}",
        }

def pretty_result(result):
    print(f"--- {result['label']} ---")
    print("ok:", result["ok"])
    if result["error"]:
        print("error:", result["error"])
    if result["response"] is not None:
        pprint(result["response"])
    print()

def ensure_bucket_exists(bucket_name: str, region_name: str):
    """
    Try to create the demo bucket if it does not exist.
    Safe to run multiple times.
    """
    head = call_aws("head_bucket", s3.head_bucket, Bucket=bucket_name)
    if head["ok"]:
        return {"action": "exists", "bucket": bucket_name, "details": head}

    if region_name == "us-east-1":
        create = call_aws("create_bucket", s3.create_bucket, Bucket=bucket_name)
    else:
        create = call_aws(
            "create_bucket",
            s3.create_bucket,
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region_name},
        )
    return {"action": "create_attempted", "bucket": bucket_name, "details": create}

def ensure_prefix_marker(bucket_name: str, prefix: str):
    """
    Put a zero-byte marker object so the path visibly exists in S3.
    """
    key = prefix.rstrip("/") + "/_KEEP"
    return call_aws(
        "put_object_marker",
        s3.put_object,
        Bucket=bucket_name,
        Key=key,
        Body=b"",
    )

def lakeformation_location_summary(resource_arn: str):
    return call_aws(
        "list_resources",
        lf.list_resources,
        FilterConditionList=[
            {
                "Field": "RESOURCE_ARN",
                "ComparisonOperator": "EQ",
                "StringValueList": [resource_arn],
            }
        ],
    )

def grant_permissions(principal_arn: str, resource: dict, permissions: list, permissions_with_grant_option=None):
    payload = {
        "Principal": {"DataLakePrincipalIdentifier": principal_arn},
        "Resource": resource,
        "Permissions": permissions,
    }
    if permissions_with_grant_option:
        payload["PermissionsWithGrantOption"] = permissions_with_grant_option
    return call_aws("grant_permissions", lf.grant_permissions, **payload)

def create_data_cells_filter(database_name: str, table_name: str, filter_name: str, row_expression: str, column_names=None):
    payload = {
        "TableData": {
            "TableCatalogId": AWS_ACCOUNT_ID,
            "DatabaseName": database_name,
            "TableName": table_name,
            "Name": filter_name,
            "RowFilter": {"FilterExpression": row_expression},
        }
    }
    if column_names:
        payload["TableData"]["ColumnNames"] = column_names
    return call_aws("create_data_cells_filter", lf.create_data_cells_filter, **payload)

## 1) Lake Registration

Registering an S3 location in Lake Formation means:

- the location becomes a **governed data lake path**,
- Lake Formation can enforce permissions against tables that use that location,
- and access moves beyond raw bucket policy thinking into centralized data governance.

Put bluntly:

- **S3 bucket policy** says who can reach the bucket.
- **Lake Formation** says who can read which governed data objects described by the catalog.

In regulated environments, that distinction matters.

In [ ]:
# ---- Lake registration demo ----
bucket_status = ensure_bucket_exists(DEMO_BUCKET, AWS_REGION)
marker_status = ensure_prefix_marker(DEMO_BUCKET, DEMO_PREFIX)

print("Bucket bootstrap:")
pprint(bucket_status)
print("\nPrefix marker:")
pretty_result(marker_status)

register_result = call_aws(
    "register_resource",
    lf.register_resource,
    ResourceArn=DEMO_S3_ARN,
    UseServiceLinkedRole=True,
)

print("Lake Formation registration attempt:")
pretty_result(register_result)

print("Location lookup after registration attempt:")
location_lookup = lakeformation_location_summary(DEMO_S3_ARN)
pretty_result(location_lookup)

### Registration interpretation

If registration succeeded, Lake Formation now recognizes that S3 ARN as a managed lake location.

If the registration call reports that the resource already exists, that is also a healthy outcome for this notebook: the location is already under governance.

Common operational reasons for a failed registration attempt:

- missing `lakeformation:RegisterResource` permission,
- the caller cannot create or use the service-linked role,
- the S3 path belongs to another account,
- or the bucket was not created and the caller lacks `s3:CreateBucket`.

For production work, registration is typically automated through IaC rather than done interactively.

## 2) Database and Table Permissions

Target example: grant `SELECT` on `citi_deep_glue.alerts` to a security analyst principal.

### Permission model

Lake Formation grants are expressed as:

- **Principal**: IAM user/role or federated principal
- **Resource**: catalog / database / table / table with columns / data filter
- **Permissions**: e.g. `SELECT`, `DESCRIBE`, `ALTER`, `DROP`, `INSERT`, `DELETE`

A practical pattern is:

- grant `DESCRIBE` so the principal can discover metadata,
- grant `SELECT` only on the tables or filtered views they should query,
- keep broader mutation permissions for engineering/admin roles only.

### Compared to IAM-only

IAM alone is fine when all you need is broad service/API access.

Lake Formation becomes necessary when the question changes from:

> Can this role call Athena/Glue/S3?

to:

> Can this role see only the `alerts` table and only a safe subset of columns and rows?

In [ ]:
# ---- Inspect Glue database/table existence ----
db_result = call_aws("get_database", glue.get_database, Name=GLUE_DATABASE)
table_result = call_aws("get_table", glue.get_table, DatabaseName=GLUE_DATABASE, Name=GLUE_TABLE)

pretty_result(db_result)
pretty_result(table_result)

In [ ]:
# ---- Grant DESCRIBE on database + SELECT/DESCRIBE on table ----
database_resource = {
    "Database": {
        "CatalogId": AWS_ACCOUNT_ID,
        "Name": GLUE_DATABASE,
    }
}

table_resource = {
    "Table": {
        "CatalogId": AWS_ACCOUNT_ID,
        "DatabaseName": GLUE_DATABASE,
        "Name": GLUE_TABLE,
    }
}

grant_db_describe = grant_permissions(
    principal_arn=SECURITY_ANALYST_ROLE_ARN,
    resource=database_resource,
    permissions=["DESCRIBE"],
)

grant_table_select = grant_permissions(
    principal_arn=SECURITY_ANALYST_ROLE_ARN,
    resource=table_resource,
    permissions=["SELECT", "DESCRIBE"],
)

pretty_result(grant_db_describe)
pretty_result(grant_table_select)

## 3) Column-level Security

Now simulate a PII-sensitive scenario.

For the `alerts` table:

- allow `endpoint_id`
- allow `severity`
- deny direct access to `message`

That pattern mirrors a real governance rule where `message` may contain operational details or customer-adjacent text that should not be broadly exposed.

### Important distinction

- `DESCRIBE` lets a principal discover table metadata.
- `SELECT` on a **table-with-columns resource** lets that principal read only the permitted columns.

So, in practice:

- someone may know the table exists,
- but still be unable to select the sensitive column.

In [ ]:
# ---- Column-level permission grant ----
table_with_columns_resource = {
    "TableWithColumns": {
        "CatalogId": AWS_ACCOUNT_ID,
        "DatabaseName": GLUE_DATABASE,
        "Name": GLUE_TABLE,
        "ColumnNames": ["endpoint_id", "severity"],
    }
}

grant_column_subset = grant_permissions(
    principal_arn=SECURITY_ANALYST_ROLE_ARN,
    resource=table_with_columns_resource,
    permissions=["SELECT", "DESCRIBE"],
)

pretty_result(grant_column_subset)

print("Expected governance intent:")
print("- endpoint_id: allowed")
print("- severity   : allowed")
print("- message    : not granted in this column-scoped resource")

## 4) Row-level Filters

Lake Formation row-level restriction is usually implemented with a **Data Cells Filter**.

For this notebook, create a filter for the analyst role so the role only sees:

```sql
severity = 'CRITICAL'
```

That is the right mental model for cases like:

- security analysts should only see critical incidents,
- regional analysts should only see `region = 'EMEA'`,
- or business users should only see data for their legal entity.

A Data Cells Filter can combine:

- a row predicate,
- and optionally a column subset,
- then be granted as the resource instead of the raw table.

In [ ]:
# ---- Create / update data cells filter for CRITICAL alerts ----
FILTER_NAME = "critical_alerts_only"

# Attempt delete first so the notebook can be rerun cleanly.
delete_filter_result = call_aws(
    "delete_data_cells_filter",
    lf.delete_data_cells_filter,
    TableCatalogId=AWS_ACCOUNT_ID,
    DatabaseName=GLUE_DATABASE,
    TableName=GLUE_TABLE,
    Name=FILTER_NAME,
)
pretty_result(delete_filter_result)

create_filter_result = create_data_cells_filter(
    database_name=GLUE_DATABASE,
    table_name=GLUE_TABLE,
    filter_name=FILTER_NAME,
    row_expression="severity = 'CRITICAL'",
    column_names=["endpoint_id", "severity", "created_at"],
)
pretty_result(create_filter_result)

get_filter_result = call_aws(
    "get_data_cells_filter",
    lf.get_data_cells_filter,
    TableCatalogId=AWS_ACCOUNT_ID,
    DatabaseName=GLUE_DATABASE,
    TableName=GLUE_TABLE,
    Name=FILTER_NAME,
)
pretty_result(get_filter_result)

In [ ]:
# ---- Grant SELECT on the row/column filter to the analyst role ----
data_cells_filter_resource = {
    "DataCellsFilter": {
        "TableCatalogId": AWS_ACCOUNT_ID,
        "DatabaseName": GLUE_DATABASE,
        "TableName": GLUE_TABLE,
        "Name": FILTER_NAME,
    }
}

grant_filter_select = grant_permissions(
    principal_arn=SECURITY_ANALYST_ROLE_ARN,
    resource=data_cells_filter_resource,
    permissions=["SELECT", "DESCRIBE"],
)

pretty_result(grant_filter_select)

## 5) LF vs IAM — decision table

| Situation | IAM alone sufficient? | Lake Formation required? | Why |
|---|---:|---:|---|
| Simple app needs broad read/write to its own S3 bucket | Yes | No | Service-level permission is enough |
| Glue crawler/admin automation managing catalog objects | Usually | Sometimes | Depends on whether governed tables are involved |
| Athena users need table-level access only | Sometimes | Usually | LF centralizes table grants better than raw IAM |
| Analysts need only certain columns | No | Yes | IAM cannot express column-level data access cleanly |
| Analysts need only certain rows | No | Yes | Row filters/Data Cells Filters are LF governance features |
| Centralized governed access across Glue/Athena/EMR | No | Yes | LF provides shared policy model and auditing |
| Highly regulated data lake with PII/need-to-know access | No | Yes | Fine-grained governance is the point |

### Hybrid mode pitfall

A common failure mode is **mixed governance assumptions**:

- some access still comes from S3/IAM,
- some access is expected to come from Lake Formation,
- and teams are not clear which control plane is authoritative.

Result: confusing "AccessDenied" behavior and policy drift.

For governed lake datasets, pick an intentional model:

- pure IAM for simple/un-governed cases, or
- Lake Formation as the governance layer for Glue/Athena/S3-backed lake resources.

In [ ]:
# ---- Optional visibility: show the current principal's Lake Formation permissions, if allowed ----
list_permissions_result = call_aws(
    "list_permissions",
    lf.list_permissions,
    Principal={"DataLakePrincipalIdentifier": SECURITY_ANALYST_ROLE_ARN},
    ResourceType="TABLE",
)

pretty_result(list_permissions_result)

## 6) What Just Happened

Lake Formation is Citi's governance layer for S3-based data lakes.

It enables:

- **column masking / restriction** for PII-like fields such as `message`,
- **row-level filtering** for severity or region isolation,
- **centralized permissioning** across lake consumers,
- and **auditable governance** suitable for regulated financial environments.

In one sentence:

> IAM answers **who can call the service**.  
> Lake Formation answers **what exact data they can see once they get there**.

In [ ]:
# ---- Final operational summary ----
summary = {
    "aws_profile_requested": AWS_PROFILE,
    "aws_region": AWS_REGION,
    "aws_account_id": AWS_ACCOUNT_ID,
    "lake_location": DEMO_S3_URI,
    "glue_database": GLUE_DATABASE,
    "glue_table": GLUE_TABLE,
    "security_analyst_role": SECURITY_ANALYST_ROLE_ARN,
    "column_level_policy": {
        "allowed_columns": ["endpoint_id", "severity"],
        "restricted_columns": ["message"],
    },
    "row_filter": {
        "filter_name": FILTER_NAME,
        "expression": "severity = 'CRITICAL'",
        "projected_columns": ["endpoint_id", "severity", "created_at"],
    },
    "postgres_source_context": POSTGRES_CONTEXT,
}

print(json.dumps(summary, indent=2))